# HuggingFace Datasets 数据集下载指南

本 Notebook 演示如何使用 [Hugging Face `datasets`](https://huggingface.co/docs/datasets) 库下载、查看与保存数据集。

**核心接口：** `datasets.load_dataset()` — 从 Hub 或本地路径加载数据集，并自动缓存到本地。

## 一、环境准备

### 1.1 安装 datasets 库

`datasets` 是 Hugging Face 官方提供的数据集加载库，需单独安装（不包含在 `transformers` 中）。

In [ ]:
# 在 Jupyter 中使用 %pip 安装 datasets 库（已安装时可注释掉本行）
%pip install datasets -q    # -q 表示“quiet”，即安装时减少输出信息（静默模式）

### 1.2 导入核心模块

| 符号 | 类型 | 说明 |
|------|------|------|
| `load_dataset` | 函数 | 从 Hub 或本地路径加载数据集，返回 `Dataset` 或 `DatasetDict` |
| `load_from_disk` | 函数 | 从 `save_to_disk` 保存的目录重新加载 |

In [ ]:
# 从 datasets 库导入 load_dataset：主入口，用于下载/加载数据集
from datasets import load_dataset

# 从 datasets 库导入 load_from_disk：从本地磁盘目录加载已保存的数据集
from datasets import load_from_disk

## 二、使用 `load_dataset` 下载数据集

### 2.1 下载公开数据集（最简用法）

**函数签名（简化）：**

```python
load_dataset(path: str, name: str = None, split: str = None, **kwargs)
```

| 参数 | 类型 | 含义 |
|------|------|------|
| `path` | `str` | Hub 上的数据集 ID，格式为 **`组织名/数据集名`**，如 `"stanfordnlp/imdb"`、`"rajpurkar/squad"`（对应页面 URL `huggingface.co/datasets/组织名/数据集名`） |
| `name` | `str`, 可选 | 子配置名（部分数据集有多个子任务/语言版本） |
| `split` | `str`, 可选 | 数据划分，如 `"train"`、`"test"`；不指定则返回全部划分 |

**返回值：** `DatasetDict`（含多个 split）或 `Dataset`（仅指定了一个 split）

In [ ]:
# 调用 load_dataset，path 为 Hub 完整数据集 ID "stanfordnlp/imdb"（组织名/数据集名）
# 不指定 split 时，返回 DatasetDict，包含 train / test / unsupervised 等划分
dataset_dict = load_dataset("stanfordnlp/imdb")

# 打印数据集对象，可看到各 split 名称与样本数量
print(dataset_dict)

### 2.2 查看数据集结构与样本

`Dataset` 类似表格：每列是一个特征（字段），每行是一条样本。

In [ ]:
# 从 DatasetDict 中取出 "train" 划分，类型为 Dataset
train_dataset = dataset_dict["train"]

# 查看列名（特征名），返回 list[str]，如 ["text", "label"]
print("特征列:", train_dataset.column_names)

# 查看数据集总样本数，返回 int
print("训练集样本数:", len(train_dataset))

# 取第 0 条样本，返回 dict，键为列名、值为对应字段
sample = train_dataset[0]
print("第 0 条样本:", sample)

### 2.3 指定配置（config）与数据划分（split）

部分数据集包含多个子配置，需通过第二个位置参数 `name` 指定；`split` 可只下载需要的划分以节省时间与磁盘。

In [ ]:
# path="nyu-mll/glue" 为 GLUE 基准套件，name="mrpc" 指定其中的 MRPC 子任务（释义等价判断）
# split="train" 只下载训练集，返回单个 Dataset 而非 DatasetDict
mrpc_train = load_dataset("nyu-mll/glue", "mrpc", split="train")

# 打印该 Dataset 的样本数量与列名
print(f"MRPC 训练集: {len(mrpc_train)} 条, 列: {mrpc_train.column_names}")

# split 也支持切片语法，只取前 100 条（适合快速调试，无需下载全量）
mrpc_subset = load_dataset("nyu-mll/glue", "mrpc", split="train[:100]")
print(f"子集样本数: {len(mrpc_subset)}")

## 三、高级下载选项

### 3.1 指定缓存目录

默认缓存路径为 `~/.cache/huggingface/datasets`。可通过 `cache_dir` 参数自定义。

| 参数 | 类型 | 含义 |
|------|------|------|
| `cache_dir` | `str`, 可选 | 数据集下载与 Arrow 缓存的本地目录 |

In [ ]:
# 导入 os，用于拼接本地路径
import os

# 定义自定义缓存目录（相对于当前工作目录）
cache_dir = os.path.join(".", "hf_datasets_cache")

# cache_dir 指定下载文件存放位置；split 只取训练集前 50 条做演示
cached_dataset = load_dataset(
    "stanfordnlp/imdb",              # path: Hub 完整数据集 ID（组织名/数据集名）
    split="train[:50]",              # split: 训练集前 50 条
    cache_dir=cache_dir,             # cache_dir: 自定义缓存路径
)

# 打印样本数，确认加载成功
print(f"已缓存加载 {len(cached_dataset)} 条样本，缓存目录: {os.path.abspath(cache_dir)}")

### 3.2 流式加载（streaming）

适合超大数据集：边下载边迭代，无需将全部数据载入内存或磁盘。

| 参数 | 类型 | 含义 |
|------|------|------|
| `streaming` | `bool` | `True` 时返回可迭代的 `IterableDataset`，而非一次性加载的 `Dataset` |

In [ ]:
# streaming=True 启用流式模式，返回 IterableDataset，不会一次性下载全量数据
stream_dataset = load_dataset("stanfordnlp/imdb", split="train", streaming=True)

# 用 iter() 获取迭代器，逐条读取样本（按需从网络拉取）
iterator = iter(stream_dataset)

# next() 取下一条样本，类型为 dict
first_sample = next(iterator)

# 打印第一条流式样本的 text 字段前 80 个字符
print("流式读取第 1 条:", first_sample["text"][:80], "...")

### 3.3 下载私有或受限数据集（需 Token）

若数据集为 **gated**（需同意协议）或 **private**，需先在 [Hugging Face 设置页](https://huggingface.co/settings/tokens) 创建 Access Token，再登录后下载。

| 方式 | 说明 |
|------|------|
| `huggingface-cli login` | 命令行交互式登录，Token 写入本地 |
| `huggingface_hub.login(token=...)` | 在代码中传入 Token（勿将 Token 提交到 Git） |
| `load_dataset(..., token=True)` | 使用已保存的 Token 访问受限资源 |

In [ ]:
# 从 huggingface_hub 导入 login：用于在 Notebook 中完成身份认证
from huggingface_hub import login

# 方式一：交互式登录（运行后会提示输入 Token，Token 会安全保存在本地）
# login()

# 方式二：从环境变量读取 Token（推荐，避免硬编码）
# import os
# login(token=os.environ["HF_TOKEN"])

# 下载 gated/私有数据集时，在 load_dataset 中传入 token=True（使用已登录的凭据）
# private_dataset = load_dataset("your-org/private-dataset", token=True)

print("私有数据集下载：先 login()，再 load_dataset(..., token=True)")

## 四、保存数据集到本地

### 4.1 使用 `save_to_disk` 持久化

将 `Dataset` / `DatasetDict` 保存为 Arrow 格式目录，便于离线复用或分享给他人。

In [ ]:
# 定义本地保存路径
save_path = os.path.join(".", "data", "imdb_subset")

# 只取 imdb 训练集前 200 条作为示例子集
subset = load_dataset("stanfordnlp/imdb", split="train[:200]")

# save_to_disk(path)：将 Dataset 序列化到 path 目录（无返回值）
subset.save_to_disk(save_path)

print(f"已保存到: {os.path.abspath(save_path)}")

### 4.2 使用 `load_from_disk` 从本地加载

无需再次联网，直接从 `save_to_disk` 生成的目录读取。

In [ ]:
# load_from_disk(path)：从本地目录加载，返回 Dataset 或 DatasetDict
loaded_subset = load_from_disk(save_path)

# 验证样本数与列名是否与保存前一致
print(f"从磁盘加载: {len(loaded_subset)} 条, 列: {loaded_subset.column_names}")

## 五、常用参数速查

| 场景 | 示例代码 |
|------|----------|
| 下载完整数据集 | `load_dataset("rajpurkar/squad")` |
| 指定子配置 | `load_dataset("nyu-mll/glue", "mrpc")` |
| 只下载某个 split | `load_dataset("stanfordnlp/imdb", split="train")` |
| 下载子集（调试用） | `load_dataset("stanfordnlp/imdb", split="train[:1000]")` |
| 自定义缓存目录 | `load_dataset("stanfordnlp/imdb", cache_dir="./cache")` |
| 流式读取（省内存） | `load_dataset("stanfordnlp/imdb", streaming=True)` |
| 私有/gated 数据集 | `load_dataset("your-org/dataset-name", token=True)` |
| 保存到本地 | `dataset.save_to_disk("./data")` |
| 从本地加载 | `load_from_disk("./data")` |

更多数据集可在 [Hugging Face Hub 数据集页](https://huggingface.co/datasets) 搜索，将页面 URL 中的 **`组织名/数据集名`** 填入 `load_dataset` 的第一个参数即可。